In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from etl.common import init_spark3

import sys
import time
import importlib
import pandas as pd

BASE_DIR = Path("/apps/jupyter/users/nguyetnguyen/features")

sys.path[:0] = [str(path) for path in [
                    BASE_DIR / "common" / "src",
                    BASE_DIR / "device_refactored",
                ]
]

from common.merge_features import merge_features_from_sample_path
from common.utils import dir_ls
from common.agg_columns import (
    build_groupby_window_query,
    build_aggregate_columns,
    get_lxw_windows,
    get_lxw_overlap_windows,
    get_feature_columns,
)
from src.device_helpers import (
    register_temp_view
)

import configs.configs as config
importlib.reload(config)

spark = init_spark3.setup(
    job_cfg={
        'executor.instances': 8,
        'executor.cores': 8,
        'executor.memory': '20g',
    },
    script_name="build_device_features_with_nid"
)

In [ ]:
# ====================
# DATA SOURCE BUILDERS
# ====================
def df_pn_nid_raw_monthly():
    return (
        spark.read.parquet(config.NID_RAW_PATH)
        .filter((F.col("date") > config.START_DATE_NID_FEATURES)
                & (F.col("date") <= config.SNAPSHOT_DATE_STR)
        )
    )

def df_pn_last_activated_date():
    return (
        spark.read.parquet(config.LAST_ACTIVATED_PATH)
        .filter((F.col("date") > config.START_DATE_LAST_ACTIVATED_DATE)
                & (F.col("date") <= config.SNAPSHOT_DATE_STR)
        )
        .groupBy("phone_number")
        .agg(F.max("last_activated_date").alias("last_activated_date"))
    )

def df_pn_nid_monthly():
    return (
        df_pn_nid_raw_monthly()
        .join(df_pn_last_activated_date(), "phone_number")
        .where("date > last_activated_date")
        .drop("last_activated_date")
    )

# ===================
# NID FEATURE BUILDERS
# ===================
def df_nid_shared_pn_counts_lxw():
    source = register_temp_view(df_pn_nid_monthly(), "df_pn_nid_monthly")
    stats = [("count(distinct phone_number)", "nid_distinct_pn_count")]
    query, _ = build_groupby_window_query(
        agg_exprs=stats,
        windows=get_lxw_windows(config.WINDOW_WEEKS),
        snapshot_date=config.SNAPSHOT_DATE_STR,
        group_by=["naid"],
        source=source,
    )
    return spark.sql(query)

def df_pn_nid_distinct_counts_lxw():
    source = register_temp_view(df_pn_nid_monthly(), "df_pn_nid_monthly")
    stats = [
        ("count(distinct naid)", "nid_distinct_nid_count"),
    ]
    
    query, _ = build_groupby_window_query(
        agg_exprs=stats,
        windows=get_lxw_windows(config.WINDOW_WEEKS),
        snapshot_date=config.SNAPSHOT_DATE_STR,
        group_by=["phone_number"],
        source=source,
    )
    return spark.sql(query)

def df_pn_by_nid_event_stats_lxw():
    source = register_temp_view(df_pn_nid_monthly(), "df_pn_nid_monthly")
    stats = [
        ("count(distinct date)", "nid_distinct_weeks_count"),
    ]
    query, _ = build_groupby_window_query(
        agg_exprs=stats,
        windows=get_lxw_windows(config.WINDOW_WEEKS),
        snapshot_date=config.SNAPSHOT_DATE_STR,
        group_by=["phone_number", "naid"],
        source=source,
    )
    return spark.sql(query)

# ===============================
# PHONE NUMBER DISTRIBUTION FEATURES
# ===============================
def df_pn_nid_distribution_lxw():
    df_join = (
        df_pn_by_nid_event_stats_lxw()
        .join(df_nid_shared_pn_counts_lxw(), on="naid", how="left")
    )
    
    source = register_temp_view(df_join, "df_join")
    prev_columns = get_feature_columns(df_join, exclude=["phone_number", "naid"])
    
    stat_config = [
        {"func": "min", "exclude": []},
        {"func": "max"},
        {"func": "avg"},
        {"func": "std"},
        {"func": "skewness"},
        {"func": "kurtosis"},
    ]
    columns, _ = build_aggregate_columns(prev_columns, stat_config)
    
    query = f"select phone_number, {', '.join(columns)} from {source} group by phone_number"
    return spark.sql(query)

# ==============
# FINAL FEATURES
# ==============
def df_pn_nid_features_lxw():
    df = (
        df_pn_nid_distribution_lxw()
        .join(df_pn_nid_distinct_counts_lxw(), on="phone_number", how="outer")
    )
    
    return df.select([
        F.when(F.isnan(c), None).otherwise(F.col(c)).alias(c)
        if t in ('double', 'float') else F.col(c)
        for c, t in df.dtypes
    ])